# Complexity-06 — Simuler un circuit quantique classiquement

**Navigation** : [<< Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling.html) | [Index de la série](README.md) | [Complexity-06b — Aaronson-Gottesman, stabilisation et déquantification >>](Complexity-06b-AaronsonGottesman-Dequantification-Python.html)

> **Public : Licence.** Le pont avec la déquantification et la stabilisation de Gottesman est
> approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.html) ;
> celui-ci s'arrête au seuil où la simulation classique devient intraitable (#P-dur).

## Ce que ce notebook suppose

- [05 — Compter est plus dur que vérifier : #P et la permanente](Complexity-05-CountingHarder-Permanent.ipynb) :
  la classe **#P**, la distinction entre **décider** et **compter**, et la formule de Ryser ;
- [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling.html) :
  le **boson sampling** comme dispositif physique qui résout #P-comptement sans algorithme classique ;
- manipuler une matrice de Pauli et un produit tensoriel.

## Ce que vous saurez faire à la fin

1. énoncer ce que signifie **simuler classiquement** un circuit quantique (sortie
   probabiliste sur n qubits = 2^n amplitudes complexes) ;
2. donner un **invariant de Pauli** sous tension : un circuit Clifford (H, S, CNOT) sur n
   qubits se décrit par **O(n²)** phases et non 2^n amplitudes — algorithme de Gottesman-Knill ;
3. **mesurer** la frontière entre la simulation polynomiale (Clifford) et la simulation
   exponentielle (universel) sur un banc classique, et relier cette frontière au statut
   **BQP ⊆ PP ⊆ P#P** (theoreme ; l’ouverte est BQP vs P / PH) ;
4. distinguer deux régimes de coût pour la simulation classique : **échantillonner** vs
   **calculer la distribution de probabilité exacte** (Aaronson-Arkhipov 2011 pour le boson sampling ; folklore pour les circuits universels) ;
5. énoncer ce que la **simulation classique** d'un circuit quantique universel nous
   apprend sur la frontière BQP/P.

**Durée estimée** : 45 minutes.

## 1. Ce que simuler classiquement signifie

Un circuit quantique à k portes et n qubits prépare un état

> |ψ⟩ = U_k · U_{k-1} · ... · U_1 · |0^n⟩

où chaque U_i est une porte (Hadamard, phase, CNOT, Toffoli…). L'état est un vecteur de
2^n amplitudes complexes ; la mesure du registre de n qubits renvoie une loi categorielle sur les 2^n chaînes de bits possibles
(la mesure d'un qubit seul projete sur la base computationnelle donne bien une Bernoulli à 2 issues,
mais c'est la mesure du registre qui est sur 2^n issues). **Simuler classiquement** le circuit, c'est
produire des échantillons de cette distribution — ou, plus exigeant, calculer exactement
les 2^n probabilités.

La question de la **simulation polynomiale** d'un circuit quantique arbitraire est l'une
des grandes questions ouvertes de la théorie : si elle est possible en P, alors BQP ⊂ P ;
si elle demande un coût exponentiel, BQP peut strictement contenir P.

Ce notebook explore une **fenêtre** dans cette question : la famille **Clifford** (Hadamard,
phase, CNOT) — où la simulation classique redevient polynomiale par l'algorithme de
Gottesman-Knill (1998). En dehors de Clifford, on entre dans la zone où la simulation
naïve coûte O(2^n).

In [1]:
import time
import numpy as np
from math import factorial

print("numpy", np.__version__)
print("python", __import__("sys").version_info[:3])

numpy 2.2.6
python (3, 11, 9)


## 2. La barrière exponentielle : simuler naïvement

Un circuit à n qubits a 2^n amplitudes. Pour un circuit universel arbitraire (Hadamard +
T + CNOT par exemple), simuler **fidèlement** chaque amplitude coûte au minimum 2^n
flops pour un produit matrice-vecteur creux, et O(2^n · profondeur) pour la totalité du
circuit. Pour la **mesure** sur un registre de n qubits, l'algorithme le plus naïf accumule
toutes les amplitudes au carré ; même les circuits de profondeur constante deviennent
intraitable au-delà de n ≈ 30.

Mesurons cette croissance : combien de temps demande une simulation vectorielle pour un
circuit de Hadamard uniquement (préparation d'une superposition uniforme) à n = 4, 8, 16,
20 qubits ?

In [2]:
def simuler_hadamard_naive(n_qubits, n_echantillons=1024):
    """Prepare |+>^n (Hadamard sur chaque qubit), mesure uniformement sur n qubits.

    Chrono englobant : construction du vecteur d'etat (coût O(2^n) en memoire)
    + calcul des probabilites + echantillonnage. C'est le coût global de la
    simulation naîve qu'on veut mesurer, pas seulement le tirage final.
    """
    dim = 1 << n_qubits
    rng = np.random.default_rng(seed=20261007)
    # Warm-up isole : exclut le coût de chargement de numpy / cache CPU
    _ = rng.choice(dim, size=64, p=np.ones(dim) / dim)
    t0 = time.perf_counter()
    # Construction du vecteur d'etat : O(2^n) memoire
    etat = np.ones(dim, dtype=np.complex128) / np.sqrt(dim)
    # Calcul des probabilites (autre O(2^n) memoire pour probas float64)
    probas = np.abs(etat) ** 2
    # Echantillonnage selon la distribution
    indices = rng.choice(dim, size=n_echantillons, p=probas)
    t1 = time.perf_counter()
    return t1 - t0

for n in (4, 8, 16, 20):
    t = simuler_hadamard_naive(n, n_echantillons=4096)
    print(f"n = {n:>2} qubits : {t*1000:7.2f} ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = {1<<n})")

n =  4 qubits :    0.67 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 16)
n =  8 qubits :    0.25 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 256)
n = 16 qubits :    1.91 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 65536)
n = 20 qubits :   24.71 ms pour 4096 echantillons (chrono complet : construction etat + echantillonnage, dim 2^n = 1048576)


Le coût mémoire du vecteur d'état croît comme 2^n nombres complexes (16 octets par
amplitude sur float64). Au-delà de n = 30 environ, la mémoire d'un poste de travail
standard (16 Go) déborde (2^30 × 16 o ≈ 17,2 GiB pour le vecteur d'état
seul ; les probas float64 et la marge système portent la limite pratique
vers n ≈ 28) ; au-delà de n = 40, il faut une machine HPC.

Cette barrière est ce qui rend les circuits quantiques **intéressants** algorithmiquement :
pour n suffisamment grand, le coût classique dépasse ce qu'on sait calculer, alors que le
même circuit reste physiquement exécutable (sur un processeur quantique réel). L'enjeu
théorique est : **quelle est la classe** des circuits qu'on peut simuler en **polynomial**
classique ?

## 3. Le résultat de Gottesman-Knill : Clifford se simule en O(n²)

Une famille de portes joue un rôle particulier en informatique quantique : les portes
**Clifford** (Hadamard, phase S, CNOT). Aaronson & Gottesman (2004) ont montré qu'un
circuit construit uniquement avec ces portes se simule classiquement en **polynomial**,
plus précisément en **O(n²)** pour la stabilisation de l'état et **O(n³)** pour
l'extraction d'une distribution d'échantillons. L'idée : l'état d'un circuit Clifford
peut être décrit non pas par ses 2^n amplitudes, mais par une **matrice de Pauli** sur
n qubits — un objet de taille O(n²) sur {I, X, Y, Z}^n.

Vérifions cette affirmation sur un cas concret : un circuit à n qubits alternant couches
de Hadamard et de CNOT, dont la distribution de probabilité de mesure finale est **uniforme**
(à un rephasage près) — donc facile à simuler par échantillonnage direct.

In [3]:
def echantillonnage_direct(n_qubits, n_echantillons=4096):
    """Echantillonnage direct d'une distribution connue analytiquement (uniforme).

    **Ce n'est PAS Gottesman-Knill** : on ne maintient pas de table de
    stabilisateurs, on n'alterne pas de portes Clifford. La fonction tire
    directement n bits i.i.d. uniformes, ce qui equivaut a mesurer
    independamment chaque qubit d'un etat |+>^n (apres un rephasage).

    Gottesman-Knill PROPREMENT DIT (matrice de Pauli O(n^2) sur n qubits,
    O(n^3) pour l'extraction d'une distribution d'echantillons) est implemente
    dans le carnet 06b (Aaronson-Gottesman, simulation par stabilisateurs) ;
    ici on ne mesure que le coût de l'echantillonnage direct d'une distribution
    CONNUE, qui sert de point de comparaison (O(n) par echantillon).
    """
    rng = np.random.default_rng(seed=20261007)
    # Warm-up isole
    _ = rng.integers(0, 2, size=(64, n_qubits), dtype=np.int8)
    t0 = time.perf_counter()
    bits = rng.integers(0, 2, size=(n_echantillons, n_qubits), dtype=np.int8)
    t1 = time.perf_counter()
    return t1 - t0, bits

for n in (4, 8, 16, 20, 24, 28):
    t, _ = echantillonnage_direct(n, n_echantillons=4096)
    print(f"n = {n:>2} qubits : {t*1000:7.2f} ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)")

n =  4 qubits :    0.06 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n =  8 qubits :    0.05 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 16 qubits :    0.10 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 20 qubits :    0.11 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 24 qubits :    0.13 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)
n = 28 qubits :    0.16 ms pour 4096 echantillons (echantillonnage direct : O(n), distribution connue)


Le coût est **quasi-constant** par qubit dans cette implémentation directe : aucune
construction de 2^n amplitudes, aucune matrice 2^n × 2^n. Le résultat Aaronson-Arkhipov (boson sampling, 2011) que
nous verrons à l'exercice 2 dit précisément quand cette stratégie reste valide, et
quand il faut payer un coût supplémentaire pour capturer des corrélations entre qubits
non triviales.

## 4. Au-delà de Clifford : la porte T et la question BQP/P

Ajouter une seule porte **non-Clifford** (typiquement la porte T = diag(1, exp(iπ/4)))
suffit à rendre le circuit **universel**. La question de la simulation polynomiale d'un
tel circuit est précisément la **frontière BQP/P** :

    BQP ⊆ P   ?   (ouvert)

où **BQP** (Bounded-Error Quantum Polynomial time) est la classe des problèmes qu'un
ordinateur quantique résout en temps polynomial avec probabilité d'erreur bornée. Si
BQP ⊂ P, alors toute la puissance du calcul quantique est simulable classiquement ; sinon,
il existe des problèmes que le quantique résout et le classique ne sait pas résoudre en
polynomial.

Le meilleur algorithme classique connu pour un circuit quantique arbitraire à n qubits
et profondeur p est en **O(2^n · poly(n, p))** — sans avancée algorithmique, cette
barrière reste exponentielle. C'est cette barrière qui motive la promesse algorithmique
du quantique ; c'est aussi elle qui rend la simulation classique **utilisable** sur les
cas Clifford, et **inutilisable** sur les cas universels.

## Exercices

Les trois exercices suivent le canevas : **mesure → compréhension → conjecture**. Le
lecteur complète l'implémentation et consigne observations et chiffrements dans les
cellules marquées `# TODO etudiant`.

### Exercice 1 — Décompte #P de la simulation universelle

On a vu qu'un circuit quantique produit une distribution de mesure sur 2^n chaînes. Pour
un circuit universel arbitraire, **calculer exactement cette distribution** est #P-dur
en général (le carnet [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling.html)
le démontre pour le boson sampling via la permanente).

**Travail** :
1. Choisir un circuit jouet de 4 qubits, universel (H, T, CNOT) ;
2. Construire la matrice unitaire complète (taille 16 × 16) ;
3. Calculer les 16 probabilités en sortie (vecteur d'état × mesure) ;
4. Vérifier que la somme des probabilités vaut 1, et observer la **distribution**
   (est-elle proche de l'uniforme, piquée sur quelques chaînes, autre ?) ;
5. Comparer au temps de calcul de la simulation Gottesman-Knill sur ce même circuit
   (cas dégénéré : pas de porte T → la simulation polynomiale doit suffire).

Notez la **frontière** : dès qu'une seule porte T est introduite, la simulation Gottesman-Knill
ne s'applique plus, et le coût redevient O(2^n).

### Exercice 2 — Distinction échantillonner / calculer (Aaronson-Arkhipov)

**Travail** :
1. Lire (ou résumer) le résultat Aaronson-Arkhipov (2011) : la distinction
   **échantillonner vs calculer** pour le boson sampling est le paradigme canonique
   de la frontière calculable / échantillonnable classiquement ; pour un circuit
   quantique UNIVERSEL sur n qubits, **calculer la distribution exacte** est
   généralement #P-dur (cf carnet 05b), tandis qu'**échantillonner** peut rester
   faisable en O(n) par échantillon si la structure du circuit le permet ;
2. Implémenter une variante Bayes-naïve sur le circuit jouet de l'exercice 1 ; mesurer le
   temps pour 1000 échantillons par rapport à l'échantillonnage direct sur Clifford ;
3. Conjecturer : pour quel type de problème (échantillonner vs calculer la distribution)
   la simulation classique est-elle acceptable en pratique ?

### Exercice 3 — La frontière BQP/P

**Travail** :
1. Énoncer (en deux phrases) la définition de BQP ;
2. Rappeler (en une phrase) ce que la barrière O(2^n) classique nous dit sur
   la séparation BQP vs P / PH (rappel : BQP ⊆ PP ⊆ P^#P est un théorème ;
   **l'ouverte** est BQP vs P / PH, pas l'inclusion BQP ⊂ P^#P) ;
3. Conjecturer (en trois phrases maximum) : si demain un algorithme classique en
   **O(poly(n))** pour les circuits universels était trouvé, que deviendrait la valeur
   algorithmique du calcul quantique ?

In [4]:
# === Cellule de verification pour les exercices ===
#
# Les trois cellules ci-dessous sont les points d'entree pour tester vos implementations.
# Elles DOIVENT s'executer sans erreur meme si les implementations sont incompletes :
# chaque bloc TODO rend None : le harnais rapporte alors l'absence de solution
# au lieu de lever une exception volontaire (regle C.1).

import sys

def _safe_run(label, fn):
    try:
        resultat = fn()
        if resultat is None:
            print(f"[todo]   {label} : a completer")
        else:
            print(f"[ok]     {label}")
    except Exception as e:
        print(f"[err]    {label} : {type(e).__name__}: {e}")


# --- Exercice 1 : distribution d'un circuit universel a 4 qubits ---
def exercice_1():
    # TODO etudiant : construire un circuit H, T, CNOT sur 4 qubits, calculer la matrice
    # unitaire 16x16 et la distribution de probabilite de mesure sur la base
    # computationnelle.
    return None  # TODO etudiant

_safe_run("Exercice 1 - distribution circuit universel 4 qubits", exercice_1)


# --- Exercice 2 : cout Best/Schoen vs Gottesman-Knill ---
def exercice_2():
    # TODO etudiant : comparer les couts d'echantillonnage entre la simulation directe
    # Gottesman-Knill (Clifford) et une variante Bayes-naive sur circuit universel.
    return None  # TODO etudiant

_safe_run("Exercice 2 - Best/Schoen vs Gottesman-Knill", exercice_2)


# --- Exercice 3 : enonce BQP, conjecturer la frontiere BQP/P ---
def exercice_3():
    # TODO etudiant : formuler (en texte) la definition de BQP, rappeler la barriere
    # O(2^n) et conjecturer sur l'avenir algorithmique du calcul quantique.
    return None  # TODO etudiant

_safe_run("Exercice 3 - BQP / P", exercice_3)

[todo]   Exercice 1 - distribution circuit universel 4 qubits : a completer
[todo]   Exercice 2 - Best/Schoen vs Gottesman-Knill : a completer
[todo]   Exercice 3 - BQP / P : a completer


## Conclusion

Ce carnet a posé la frontière de la simulation classique d'un circuit quantique par le
biais de deux résultats canoniques :

- la **barrière exponentielle** O(2^n) d'un circuit universel arbitraire, qui sépare la
  zone où la simulation classique naïve reste praticable (n petit) de celle où elle
  devient intraitable (n ≥ 30) ;
- l'**algorithme de Gottesman-Knill** (1998), qui montre que la sous-famille **Clifford**
  (H, S, CNOT) reste simulable en O(n²) — polynomiale, donc compatible avec une
  simulation classique exhaustive.
Le pont avec la **déquantification Aaronson-Gottesman** et la **stabilisation** est
approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.html).

**Sources** :
- Aaronson & Arkhipov, *The Computational Complexity of Linear Optics*, Theory of Computing 9 (2013), pp. 143-252 (boson sampling : distinction echantillonner / calculer) ;
- Aaronson & Gottesman, *Improved Simulation of Stabilizer Circuits*, Phys. Rev. A 70 (2004) ;
- Nielsen & Chuang, *Quantum Computation and Quantum Information*, 10th anniv. ed. (2010),
  ch. 10 sur la stabilité et la simulation ;
- Aaronson, *BQP and the Polynomial Hierarchy* (2009), pour la conjecture BQP ⊄ PH.